# LangChain Expression Language (LCEL)

## Name : Santhose Arockiaraj J
## RegNo: 212224230248

### Simple Chain

In [55]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

In [56]:
#!pip install pydantic==1.10.8

In [57]:
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.schema.output_parser import StrOutputParser

In [58]:
prompt = ChatPromptTemplate.from_template(
    "tell me a quote about the {topic}"
)
model = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)
output_parser = StrOutputParser()

In [59]:
chain = prompt | model | output_parser

In [60]:
chain.invoke({"topic": "Pyscoholgy"})

'"Psychology is the science of behavior and mental processes, revealing the intricate tapestry of the human experience."'

### More Complex Chain

In [61]:
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import DocArrayInMemorySearch

In [62]:
vectorstore = DocArrayInMemorySearch.from_texts(
    [
        "Dravid works at Zoho",
        "Doremon likes to eat doracake"
    ],
    embedding=OpenAIEmbeddings()
)

retriever = vectorstore.as_retriever()

In [63]:
retriever.get_relevant_documents("where did Dravid work?")

[Document(page_content='Dravid works at Zoho'),
 Document(page_content='Doremon likes to eat doracake')]

In [64]:
retriever.get_relevant_documents("what does Doremon like to eat")

[Document(page_content='Doremon likes to eat doracake'),
 Document(page_content='Dravid works at Zoho')]

In [65]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [66]:
from langchain.schema.runnable import RunnableMap

In [67]:
chain = RunnableMap({
    "context": lambda x: retriever.get_relevant_documents(x["question"]),
    "question": lambda x: x["question"]
}) | prompt | model | output_parser

In [68]:
chain.invoke({"question": "where did Dravid work?"})

'Dravid works at Zoho.'

In [ ]:
chain.invoke({"question": "what does Doremon like to eat"})

In [69]:
inputs = RunnableMap({
    "context": lambda x: retriever.get_relevant_documents(x["question"]),
    "question": lambda x: x["question"]
})

In [71]:
inputs.invoke({"question": "where did Dravid work?"})

{'context': [Document(page_content='Dravid works at Zoho'),
  Document(page_content='Doremon likes to eat doracake')],
 'question': 'where did Dravid work?'}